## Models trained on original (cleaned and normalized) COCOMO dataset

In [15]:
# Normalization of the original data 

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, confusion_matrix, accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn import svm
import numpy as np
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from skopt import BayesSearchCV
from skopt.space import Real, Integer

data = "/Users/alexandraantonica/Desktop/RW/Cleaned_data.csv"
df = pd.read_csv(data)

# Create a MinMaxScaler object
scaler = MinMaxScaler()

# Apply min-max normalization on the "augmented_data" DataFrame
normalized_data = df.copy()  # Create a copy of the DataFrame
normalized_data[['E', 'PEMi', 'Size_KLOC', 'ACT_EFFORT']] = scaler.fit_transform(df[['E', 'PEMi', 'Size KLOC', 'ACT_EFFORT']])

df = normalized_data
df

,Unnamed: 0,E,PEMi,Size KLOC,ACT_EFFORT,categorical output,shorter convetion for output,Size_KLOC
0,0,0.383287,0.253579,9.5,0.023637,small,S,0.033021
1,1,0.383287,0.253579,8.4,0.028613,small,S,0.029197
2,2,0.383287,0.253579,10.6,0.017417,small,S,0.036844
3,3,0.383287,0.253579,2.7,0.000000,small,S,0.009385
4,4,0.383287,0.253579,4.6,0.002488,small,S,0.015989
...,...,...,...,...,...,...,...,...
71,84,0.557103,0.074581,111.0,0.613311,large,L,0.385819
72,85,0.557103,0.339566,162.0,0.775036,large,L,0.563087
73,87,0.913092,0.314952,100.0,0.720091,large,L,0.347584
74,89,0.913092,0.696986,41.0,0.612275,large,L,0.142510


### Logistic Regression

In [9]:
X = df[['E', 'PEMi', 'Size KLOC', 'ACT_EFFORT']]
y = df["categorical output"]

# Splitting the data into training and test sets with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
print(X_train.shape, y_train.shape)

logreg = LogisticRegression()
logreg.fit(X_train, y_train)

# Using .score() to calculate the accuracy
train_accuracy_logreg = logreg.score(X_train, y_train )
test_accuracy_logreg = logreg.score(X_test, y_test)

# Predictions for calculating recall
y_test_pred = logreg.predict(X_test)

# Calculate recall
test_recall_logreg = recall_score(y_test, y_test_pred, average='macro')

# Compute confusion matrix
conf_matrix_lr = confusion_matrix(y_test, y_test_pred)

# Print the metrics

print(f"Training score: {train_accuracy_logreg:.3f}")
print(f"Test set score: {test_accuracy_logreg:.3f}")
print(f"Testing Recall: {test_recall_logreg:.3f}")
print("Confusion Matrix:") 
print(conf_matrix_lr)


(57, 4) (57,)
Training score: 0.719
Test set score: 0.737
Testing Recall: 0.556
Confusion Matrix:
[[ 2  1  0]
 [ 0 12  0]
 [ 0  4  0]]


/Users/alexandraantonica/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### SVM

In [14]:
X = df[['E', 'PEMi', 'Size KLOC', 'ACT_EFFORT']]
y = df["categorical output"]

# Splitting the data into training and test sets with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

classifier = svm.SVC(kernel='linear', gamma="auto", C=2)
classifier.fit(X_train, y_train)

# Making predictions on the training and test sets
y_train_pred = classifier.predict(X_train)
y_test_pred = classifier.predict(X_test)

# Calculate accuracy and recall
train_accuracy_svm = accuracy_score(y_train, y_train_pred)
test_accuracy_svm = accuracy_score(y_test, y_test_pred)
train_recall_svm = recall_score(y_train, y_train_pred, average='macro')
test_recall_svm = recall_score(y_test, y_test_pred, average='macro')

# Compute confusion matrix
conf_matrix_svm = confusion_matrix(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy_svm:.3f}")
print(f"Test Accuracy: {test_accuracy_svm:.3f}")
print(f"Training Recall: {train_recall_svm:.3f}")
print(f"Testing Recall: {test_recall_svm:.3f}")
print("Confusion matrix:")
print(conf_matrix_svm)


Training Accuracy: 0.912
Test Accuracy: 0.842
Training Recall: 0.826
Testing Recall: 0.778
Confusion matrix:
[[ 2  1  0]
 [ 1 11  0]
 [ 0  1  3]]


### XGBoost

In [16]:
# Define target and input features
y = df['categorical output']
X = df[['E', 'PEMi', 'Size KLOC', 'ACT_EFFORT']]

# Encode categorical output
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42)

# Define the search space for Bayesian optimization
param_space = {
    'learning_rate': Real(0.0001, 0.5, 'log-uniform'),
    'max_depth': Integer(3, 11),
    'alpha': Real(1e-9, 100.0, 'log-uniform'),
    'lambda': Real(1e-9, 100.0, 'log-uniform')
}

# Initialize XGBoost classifier with multi-class objective
xgb_model = XGBClassifier(objective='multi:softmax', num_class=len(np.unique(y_encoded)))

# Perform Bayesian optimization with cross-validation
bayes_search = BayesSearchCV(
    estimator=xgb_model,
    search_spaces=param_space,
    scoring='accuracy',
    cv=15,
    n_iter=50,
    random_state=42,
    n_jobs=-1
)

# Perform optimization
bayes_search.fit(X_train, y_train)

# Get best parameters and retrain the model
best_params = bayes_search.best_params_
print("Best Parameters:", best_params)

best_xgb_model = bayes_search.best_estimator_
best_xgb_model.fit(X_train, y_train)

# Predictions
train_predictions = best_xgb_model.predict(X_train)
test_predictions = best_xgb_model.predict(X_test)

# Calculate accuracy
train_accuracy_xg = accuracy_score(y_train, train_predictions)
test_accuracy_xg = accuracy_score(y_test, test_predictions)

# Calculate recall
train_recall_xg = recall_score(y_train, train_predictions, average='macro')
test_recall_xg = recall_score(y_test, test_predictions, average='macro')

# Compute confusion matrix
conf_matrix_xgboost = confusion_matrix(y_test, test_predictions)

print(f"Training Accuracy: {train_accuracy_xg:.3f}")
print(f"Testing Accuracy: {test_accuracy_xg:.3f}")
print(f"Training Recall: {train_recall_xg:.3f}")
print(f"Testing Recall: {test_recall_xg:.3f}")
print("Confusion Matrix:") 
print(conf_matrix_xgboost)


/Users/alexandraantonica/anaconda3/lib/python3.10/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 8 members, which is less than n_splits=15.
  warnings.warn(
/Users/alexandraantonica/anaconda3/lib/python3.10/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 8 members, which is less than n_splits=15.
  warnings.warn(
/Users/alexandraantonica/anaconda3/lib/python3.10/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 8 members, which is less than n_splits=15.
  warnings.warn(
/Users/alexandraantonica/anaconda3/lib/python3.10/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 8 members, which is less than n_splits=15.
  warnings.warn(
/Users/alexandraantonica/anaconda3/lib/python3.10/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated 

Best Parameters: OrderedDict([('alpha', 3.244468390655083e-05), ('lambda', 0.10115402653101345), ('learning_rate', 0.2822609154544828), ('max_depth', 6)])
Training Accuracy: 1.000
Testing Accuracy: 1.000
Training Recall: 1.000
Testing Recall: 1.000
Confusion Matrix:
[[ 3  0  0]
 [ 0 14  0]
 [ 0  0  6]]
